<img src="../img/GTK_Logo_Social Icon.jpg" width=175 align="right" />


# Worksheet 10.1: Talking to Claude Programmatically

*Module 10 — Working with LLMs.* Cells marked **TODO** are yours to write; everything else is ready to run.

So far you've *used* models. Now you'll *call* one from code. This worksheet has two parts:

1. **Anomaly detection on time-series data** — turn a table of CPU readings into a prompt and ask an LLM to flag the unusual ones, returned as structured JSON.
2. **Image in, structured data out** — send a photo of a vehicle and get back a clean JSON record (license plate, make, model, and more).

Both parts share one idea that powers most real AI systems: turning messy input (a table, an image, a log dump) into structured records your other code can act on.

## 0. Setup

We use Anthropic's official `anthropic` package, plus `pandas` for the data in Part 1:

In [ ]:
!pip install -q anthropic pandas

### Your API key

The client reads your key from the `ANTHROPIC_API_KEY` environment variable — **never paste your key into a notebook you might share.** Set it before launching Jupyter:

```bash
export ANTHROPIC_API_KEY="sk-ant-..."
```

Get a key at [console.anthropic.com](https://console.anthropic.com). If the next cell raises an auth error, your key is not set.

In [ ]:
import anthropic

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-opus-4-8"        # the model we'll call throughout

print("Client ready.")

## Part 1 — Anomaly detection with an LLM

In the Anomaly Detection worksheet you found unusual points in CPU time-series data using statistics. Here you'll ask an LLM to do the same thing from a plain-text description of the data — no model to train, just a well-built prompt.

Every request goes through `client.messages.create(...)`. You set `model`, `max_tokens`, and `messages` (the conversation, as a list of `{"role", "content"}` turns); the reply text is at `response.content[0].text`. You'll see that pattern in a helper below — then in Part 2 you'll write a call yourself.

The plan:

1. Load the CPU data — a clean "training" set with no anomalies, and a "test" set with an anomaly about 10 rows in.
2. Format the rows as text the model can read.
3. Build a prompt: some normal data for reference, then the unknown data, and a request to return the anomalies as JSON.
4. Call Claude and parse the result.

In [ ]:
import pandas as pd

training_data = pd.read_csv("../data/cpu-train-b.csv", parse_dates=["datetime"])
testing_data  = pd.read_csv("../data/cpu-test-b.csv",  parse_dates=["datetime"])

print("training rows:", len(training_data), "| testing rows:", len(testing_data))
testing_data.head()

### Step 1 — Format the data for the prompt

The model reads text, so turn each row into a line like this:

```
Timestamp: 2017-01-28 04:42:00, cpu: 1.71
```

Write a function that turns a dataframe into one string of these lines (one row per line).

In [ ]:
def format_data_for_prompt(df):
    """Turn a dataframe of (datetime, cpu) rows into one text block,
    with one 'Timestamp: <dt>, cpu: <cpu>' line per row."""
    data = ""
    for _, row in df.iterrows():
        data += f"Timestamp: {row['datetime']}, cpu: {row['cpu']}\n"
    return data

# quick check — should print three formatted lines
print(format_data_for_prompt(testing_data.head(3)))

### Step 2 — Build the prompt

Give the model a sample of normal data for reference, then the unknown rows, and tell it exactly what to return. You don't need the whole training set — a sample of ~50 rows keeps the request small and cheap. Send at least the first 20 test rows so the anomaly is included (we'll use 40).

We ask for the anomalies back as **JSON** so we can parse them in code.

In [ ]:
# 1) a sample of ~50 NORMAL rows from training_data
normal_data  = format_data_for_prompt(training_data.sample(50))
# 2) the first 40 rows of unknown data from testing_data
unknown_data = format_data_for_prompt(testing_data.head(40))

prompt = (
    f"The following are normal CPU time-series readings, for reference:\n{normal_data}\n"
    f"Below are unknown readings. Find any anomalies.\n{unknown_data}\n"
    'Return ONLY a JSON list of the anomalous rows, each shaped like '
    '{"datetime": "...", "cpu": ...}. No markdown, no commentary.'
)

print(prompt)

### Step 3 — Call Claude

Here's a small helper that sends a text prompt and returns Claude's reply text. **Read it** — this is the `client.messages.create` pattern, and in Part 2 you'll write one of these yourself (for an image).

In [ ]:
def call_claude(prompt):
    """Send a text prompt to Claude and return the reply text."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

raw = call_claude(prompt)
print(raw)

### Step 4 — Parse the anomalies

The reply is still a *string*. Turn it into real Python so your code could act on it. Models sometimes wrap JSON in a ```` ```json ```` code fence, so we strip that first, then call `json.loads()`. You'll **reuse this `extract_json` helper in Part 2**, so get it working here.

In [ ]:
import json
import re

def extract_json(text):
    """Strip an optional ```json code fence, then parse to Python."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text).strip()
    return json.loads(text)

anomalies = extract_json(raw)
print("Claude flagged", len(anomalies), "anomalous rows")
anomalies

### How did it do?

Compare the rows Claude flagged against the real anomaly — about 10 rows into the test set, where the CPU value drops close to zero. Did it catch them?

You can also ask the model *why* a point is anomalous, which makes this a handy first pass for triaging logs or metrics. The catches are the usual ones: every call costs money, and you're sending your data to a third party.

## Part 2 — Image in, structured JSON out

Same idea, different input. We'll hand Claude a photo of a vehicle and ask it to return a **JSON object** with the fields we care about — the kind of record you'd write to a database or feed to the next stage of a pipeline.

Three moves make this work:

1. **Encode the image** as base64 so it can travel inside the request.
2. **Send two content blocks** in one message: the image, then a text prompt telling Claude exactly which fields to extract and to reply with *only* JSON.
3. **Parse the reply** with the `extract_json` helper you wrote in Part 1.

We've bundled three photos in `../data/vehicles/`. First, a helper to read a file and base64-encode it:

In [ ]:
import base64
from IPython.display import Image

def to_base64(path):
    """Read an image file and return it as a base64 string."""
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")

# Take a look at the vehicle we'll analyze first:
Image(filename="../data/vehicles/suv.jpg", width=500)

### Building the request

The prompt does the heavy lifting: it lists the exact fields, says what type each should be, and tells Claude to use `null` when something can't be determined and to return **raw JSON only**. Being explicit here is what makes the output predictable.

In [ ]:
EXTRACT_PROMPT = """You are analyzing a photo of a vehicle. Extract these fields and
respond with ONLY a raw JSON object — no markdown, no commentary:

- license_plate: the plate text, or null if not readable
- vehicle_type: e.g. "sedan", "SUV", "pickup truck", "van", "motorcycle"
- make: manufacturer, or null
- model: or null
- color
- body_style: e.g. "4-door sedan", "2-door coupe utility"
- approximate_year: integer, or null
- visible_damage: short description, or null if none
- occupancy: number of people visible in the vehicle, or null

Use null for anything you cannot determine from the image."""

img_b64 = to_base64("../data/vehicles/suv.jpg")

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": img_b64,
                    },
                },
                {"type": "text", "text": EXTRACT_PROMPT},
            ],
        }
    ],
)

raw = response.content[0].text
print(raw)

### From text to a Python dict

Same story as Part 1: the reply is a JSON string. Reuse the `extract_json` helper you already wrote to parse it into a dict.

In [ ]:
vehicle = extract_json(raw)

print(type(vehicle))          # <class 'dict'> — real Python now
print("Plate:", vehicle["license_plate"])
print("Make: ", vehicle["make"])
vehicle

### TODO

Two of the bundled photos have a readable plate; one does not. Try the extractor on a different image and see how Claude reports the missing plate.

The other files are `../data/vehicles/sedan.jpg` and `../data/vehicles/pickup.jpg`.

1. Display one of them, run the extraction, and parse the result with `extract_json`.
2. **Add one field of your own** to `EXTRACT_PROMPT` (for example `"number_of_doors"` or `"is_commercial_vehicle"`) and confirm it shows up in the returned dict.

In [ ]:
# Try the pickup, which has no visible plate — Claude should return null for it.
Image(filename="../data/vehicles/pickup.jpg", width=500)

In [ ]:
# Add a field ("number_of_doors") to the prompt, then run it on the pickup.
EXTRACT_PROMPT_V2 = EXTRACT_PROMPT + "\n- number_of_doors: integer, or null"

img_b64 = to_base64("../data/vehicles/pickup.jpg")

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image",
                 "source": {"type": "base64", "media_type": "image/jpeg", "data": img_b64}},
                {"type": "text", "text": EXTRACT_PROMPT_V2},
            ],
        }
    ],
)

vehicle = extract_json(response.content[0].text)
print("Plate (expect None/null):", vehicle["license_plate"])
print("New field:", vehicle.get("number_of_doors"))
vehicle

## Recap

You just did the two things almost every LLM application is built on:

- **Part 1** — turned a table into a text prompt, asked Claude to find anomalies, and parsed the JSON answer into Python.
- **Part 2** — sent an image plus a field-list prompt and parsed the result into a `dict`.

The through-line is the *prompt contract*: when you tell the model exactly which fields to return and in what shape, you turn a chatty assistant into a reliable component you can parse with `extract_json`. Later we'll see how to make that contract even stricter so the output is guaranteed to parse.